In [1]:
import pandas as pd
import glob
import numpy as np

def audit_trading_performance():
    # On récupère tous les fichiers de log qui commencent par 'trade_log_'
    all_files = glob.glob("trade_log_*.csv")
    
    if not all_files:
        print("❌ Aucun fichier de log trouvé. Commencez d'abord par faire du Paper Trading !")
        return

    full_results = []
    
    for file in all_files:
        df = pd.read_csv(file)
        # On ne prend que la dernière ligne de chaque trade pour connaître le résultat final
        final_state = df.iloc[-1]
        full_results.append({
            'Ticker': file.split('_')[2],
            'PnL': final_state['PnL'],
            'Status': final_state['Status']
        })
    
    report = pd.DataFrame(full_results)
    
    # --- CALCUL DES MÉTRIQUES ---
    wins = report[report['PnL'] > 0]
    losses = report[report['PnL'] <= 0]
    
    win_rate = (len(wins) / len(report)) * 100 if len(report) > 0 else 0
    total_profit = wins['PnL'].sum()
    total_loss = abs(losses['PnL'].sum())
    
    # Profit Factor : $ gagnés pour chaque $ perdu (Objectif : > 1.5)
    profit_factor = total_profit / total_loss if total_loss > 0 else float('inf')
    
    # Espérance mathématique par trade
    expectancy = report['PnL'].mean()

    print("="*40)
    print("📋 AUDIT DE PERFORMANCE QUANT")
    print("="*40)
    print(f"Nombre de trades analysés : {len(report)}")
    print(f"Taux de réussite (Win Rate) : {win_rate:.2f}%")
    print(f"Profit Factor               : {profit_factor:.2f}")
    print(f"Espérance par trade         : {expectancy:+.2f}$")
    print("-" * 40)
    
    if expectancy > 0 and win_rate > 50:
        print("🚀 VERDICT : Stratégie robuste. Prêt pour un test à capital réel réduit.")
    else:
        print("⚠️ VERDICT : L'avantage statistique est trop faible. Ajustez vos seuils (PoP/Kelly).")

# Lancer l'audit
audit_trading_performance()

❌ Aucun fichier de log trouvé. Commencez d'abord par faire du Paper Trading !
